In [ ]:
from astropy.io import fits
from astropy.coordinates import SkyCoord
import numpy as np
import matplotlib.pyplot as plt
from reproject.mosaicking import find_optimal_celestial_wcs
from reproject import reproject_interp
from reproject.mosaicking import reproject_and_coadd
from astropy.wcs import WCS
from astropy.utils.data import get_pkg_data_filename
from astropy.convolution import Gaussian2DKernel
#from scipy.signal import convolve as scipy_convolve
from astropy.convolution import convolve
import gc
from matplotlib.patches import Circle
from scipy.stats import linregress

## Functions

In [ ]:
def read_files_4channels(directory,stokes,filetype):

    band = ['A','B','C','D']
    data_list = []
    hdr_list = []
    print('Reading in '+stokes+' for filetype: '+filetype)

    for i in range(0,4):
        print('band '+band[i])
        hdu = fits.open(directory+stokes+band[i]+'_'+filetype+'.fits')
        data_list.append(hdu[0].data)

        hdr = fits.Header()
        for card in hdu[0].header.cards:
            if card.keyword.strip() != "":
                hdr.append(card)
        hdr['OBJECT'] = stokes+band[i]+'_'+filetype
        hdr_list.append(hdr)
        #print(repr(hdr))
        #print('-------------------')

    gc.collect()

    return data_list,hdr_list

In [ ]:
def calc_errors(dir_in, l_list, b_list, filename, boxsize = 5):

    CG_Q_list, hdr_CG_Q_list = read_files_4channels(dir_in,'Q',filename)
    CG_U_list, hdr_CG_U_list = read_files_4channels(dir_in,'U',filename)

    lons = WCS(hdr_CG_Q_list[0]).all_pix2world(range(CG_Q_list[0].shape[1]) ,0, 0)[0]
    lats = WCS(hdr_CG_Q_list[0]).all_pix2world(0, range(CG_Q_list[0].shape[0]), 0)[1]

    dQ_list = np.empty([len(l_list),4])
    dU_list = np.empty([len(l_list),4])

    for j in range(0,len(l_list)):

        xidx = np.where(abs(lons - l_list[j]) < boxsize)[0]
        yidx = np.where(abs(lats - b_list[j]) < boxsize)[0]

        #print(lons[xidx])
        #print(lats[yidx])

        for i in range(0,4):

            Q_sub = CG_Q_list[i][yidx[0]:yidx[-1],xidx[0]:xidx[-1]]
            U_sub = CG_U_list[i][yidx[0]:yidx[-1],xidx[0]:xidx[-1]]
            print(Q_sub.shape)
            #print(U_sub.shape)

            dQ_list[j,i] = np.nanstd(Q_sub.flatten())
            dU_list[j,i] = np.nanstd(U_sub.flatten())

    del CG_Q_list
    del CG_U_list
    gc.collect()
    
    return dQ_list, dU_list

In [ ]:
def make_test_map(data,hdr,vmin=-0.5,vmax=0.5,cmap='RdBu_r',
                  *args,**kwargs):

    wcs = WCS(hdr)
    fs = 20
    plt.figure(figsize=(40,4))
    plt.subplot(projection=wcs)
    plt.imshow(data,origin='lower',vmin=vmin,vmax=vmax,cmap=cmap)
    plt.xlabel('Galactic Longitude',fontsize=fs)
    plt.ylabel('Galactic Latitude',fontsize=fs)
    plt.tick_params(labelsize=fs,axis='both')
    plt.title(hdr['OBJECT'],fontsize=fs)
    cbar = plt.colorbar(pad=0.002)
    cbar.ax.tick_params(labelsize=fs)

    gc.collect()

    return

In [ ]:
def make_PI_file(directory,Q_list,U_list,hdr,avg_order='avg_PI',keep_PI=True,savefile=False,
                 *args,**kwargs):

    hdr_PI = hdr[0].copy()
    
    filetype = hdr_PI['OBJECT'][3:]
    print(filetype)
    hdr_PI['OBJECT'] = filetype+'_'+avg_order
    
    if avg_order == 'mean_of_PI':
        # Calculate PI in each band, then take average PI across the band
        PI = np.nanmean(np.sqrt(np.array(Q_list)**2+np.array(U_list)**2),axis=0)
        
    if avg_order == 'PI_of_mean':
        # Take the averages of Q and U across the band, then calculate PI 
        PI = np.sqrt(np.nanmean(np.array(Q_list),axis=0)**2+np.nanmean(np.array(U_list),axis=0)**2)

    if savefile:
        fits.writeto(directory+'PI_'+filetype+'_'+avg_order+'.fits',PI,header=hdr_PI,overwrite=True)

    if keep_PI == False:
        del PI
        PI = 0
    gc.collect()

    return PI, hdr_PI

In [ ]:
def calc_pol_angles(Q_list,U_list,hdrQ_list,hdrU_list,errQ,errU,
                    keep_PA=True,savefile=False,*args,**kwargs):
    
    PA = []
    errPA = []
    zz = []
    dtau = []
    descrptn = []
    hdr_PA = []
    
    for i in range(0,len(Q_list)):
        
        descrptn.append('PA_'+hdrQ_list[i]['OBJECT'][1:])
        
        PA.append(0.5*np.arctan2(U_list[i],Q_list[i]))

        errPA.append( np.sqrt((errQ**2)*(U_list[i]**2) + (errU**2)*(Q_list[i]**2))/(2*(Q_list[i]**2+U_list[i]**2)) )
        
        ## correct angle wrap:
        zz.append(np.exp(2*PA[i]*1j))
        if i > 0:
            dtau.append(np.arctan2((zz[i]*np.conj(zz[i-1])).imag,
                                   (zz[i]*np.conj(zz[i-1])).real)/2.)
            PA[i] = PA[i-1]+dtau[i-1]
            
        hdr_new = hdrQ_list[i].copy()
        hdr_new['OBJECT'] = descrptn[i]

        hdu_list = fits.HDUList()   
        hdr_new['EXTNAME'] = 'PA'
        hdu1 = fits.ImageHDU(PA[i], header=hdr_new)
        hdr_new['EXTNAME'] = 'PA_err'
        hdu2 = fits.ImageHDU(errPA[i], header=hdr_new)

        for hdu_app in [hdu1,hdu2]:
            hdu_list.append(hdu_app)

        if savefile:        
            hdu_list.writeto(directory+descrptn[i]+'.fits',overwrite=True)
        
        print('finished calculating PA for array '+str(i+1)+' of '+str(len(Q_list)))
            
    #if savefile:
    #    for i in range(0,len(PA)):
    #        fits.writeto(directory+descrptn[i]+'.fits',PA[i],
    #                     header=hdr_PA[i],overwrite=True)
    
    del zz; del dtau; del errPA
    if keep_PA == False:
        del PA
        PA = 0
    gc.collect()
    
    return PA, hdr_PA

In [ ]:
def noise_map(map_list,hdr,dpix):

    noise_maps = {}

    noise_maps['qa'] = np.empty_like(map_list[0])
    noise_maps['qb'] = np.empty_like(map_list[0])
    noise_maps['qc'] = np.empty_like(map_list[0])
    noise_maps['qd'] = np.empty_like(map_list[0])
    noise_maps['ua'] = np.empty_like(map_list[0])
    noise_maps['ub'] = np.empty_like(map_list[0])
    noise_maps['uc'] = np.empty_like(map_list[0])
    noise_maps['ud'] = np.empty_like(map_list[0])

    for i in range(0,map_list[0].shape[0]):
        for j in range(0,map_list[0].shape[1]):

            noise_maps['qa'][i,j] = np.std(map_list[0][i-dpix:i+dpix,j-dpix:j+dpix])
            noise_maps['qb'][i,j] = np.std(map_list[1][i-dpix:i+dpix,j-dpix:j+dpix])
            noise_maps['qc'][i,j] = np.std(map_list[2][i-dpix:i+dpix,j-dpix:j+dpix])
            noise_maps['qd'][i,j] = np.std(map_list[3][i-dpix:i+dpix,j-dpix:j+dpix])

            noise_maps['ua'][i,j] = np.std(map_list[4][i-dpix:i+dpix,j-dpix:j+dpix])
            noise_maps['ub'][i,j] = np.std(map_list[5][i-dpix:i+dpix,j-dpix:j+dpix])
            noise_maps['uc'][i,j] = np.std(map_list[6][i-dpix:i+dpix,j-dpix:j+dpix])
            noise_maps['ud'][i,j] = np.std(map_list[7][i-dpix:i+dpix,j-dpix:j+dpix])

        print(i)
            
    return noise_maps

In [ ]:
from scipy.optimize import curve_fit

def linear_model(x, m, c):
    return m * x + c

In [ ]:
def calc_RM(PA_list,PA_err_list,hdr,keep_RM=True,savefile=False,*args,**kwargs):
    
    freq = np.array([1406.9,1413.8,1427.4,1434.3])
    lbd2 = ((3e8)/(freq*1e6))**2
    
    RM     = np.empty_like(PA_list[0])
    PAint  = np.empty_like(PA_list[0])
    rvalue = np.empty_like(PA_list[0])
    pvalue = np.empty_like(PA_list[0])
    stderr = np.empty_like(PA_list[0])
    intstd = np.empty_like(PA_list[0])
    
    descrptn_RM = 'RM_'+hdr['OBJECT'][5:]
    
    for i in range(0,RM.shape[0]):
        print(str(i)+' of '+str(RM.shape[0])+' lat pixels')
        for j in range(0,RM.shape[1]):
            if np.isfinite(PA_list[0][i,j]):
                PA = np.array([PA_list[0][i,j],PA_list[1][i,j],
                               PA_list[2][i,j],PA_list[3][i,j]])
                PA_err = np.array([PA_err_list[0][i,j],PA_err_list[1][i,j],
                                   PA_err_list[2][i,j],PA_err_list[3][i,j]])
            
                try:
                    result = linregress(lbd2,PA)
                    #RM[i,j]     = result.slope
                    #PAint[i,j]  = result.intercept
                    #rvalue[i,j] = result.rvalue
                    pvalue[i,j] = result.pvalue
                    stderr[i,j] = result.stderr
                    #intstd[i,j] = result.intercept_stderr

                    popt, pcov = curve_fit(linear_model, lbd2, PA, sigma=PA_err, absolute_sigma=True)
                    RM[i,j], PAint[i,j]      = popt
                    stderr[i,j], intstd[i,j] = np.sqrt(np.diag(pcov))

                    
                except:
                    pass
                del PA, PA_err

        gc.collect()
            
    hdr_RM = hdr.copy()
    hdr_RM['OBJECT'] = descrptn_RM
    hdu_list = fits.HDUList()
    
    hdr_RM['EXTNAME'] = 'RM'
    hdu1 = fits.ImageHDU(RM, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'PA_INT'
    hdu2 = fits.ImageHDU(PAint, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'RVALUE'
    hdu3 = fits.ImageHDU(rvalue, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'PVALUE'
    hdu4 = fits.ImageHDU(pvalue, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'STDERR'
    hdu5 = fits.ImageHDU(stderr, header=hdr_RM)
    hdr_RM['EXTNAME'] = 'INT_STDERR'
    hdu6 = fits.ImageHDU(intstd, header=hdr_RM)

    for hdu_app in [hdu1,hdu2,hdu3,hdu4,hdu5,hdu6]:
        print(hdu_app)
        hdu_list.append(hdu_app)

    if savefile:        
        hdu_list.writeto(directory+descrptn_RM+'.fits',overwrite=True)
    
    if keep_RM == False:
        del RM; del hdu_list; del PAint
        del rvalue; del pvalue; del stderr; del intstd
        RM = 0; hdu_list = 0
    gc.collect()
    
    return RM, hdu_list

## Calculate convolved, regridded PI for all (CGPS, GMIMS, CGPS+GMIMS)

In [ ]:
%%time
go = True

directory  = '/srv/data/cgps-gmims/conv_regrid/'

if go:

    G_Q_list, hdr_G_Q_list = read_files_4channels(dir_in,'Q','G_regrd')
    G_U_list, hdr_G_U_list = read_files_4channels(dir_in,'U','G_regrd')
    
    C_Q_list, hdr_C_Q_list = read_files_4channels(dir_in,'Q','C_conv4_regrd')
    C_U_list, hdr_C_U_list = read_files_4channels(dir_in,'U','C_conv4_regrd')
    
    CG_Q_list, hdr_CG_Q_list = read_files_4channels(dir_in,'Q','CG_conv4_regrd')
    CG_U_list, hdr_CG_U_list = read_files_4channels(dir_in,'U','CG_conv4_regrd')
    
    G_avg_PI, hdr_G_avg_PI = make_PI_file(directory,G_Q_list,G_U_list,hdr_G_Q_list,
                                           avg_order='mean_of_PI', keep_PI=False, savefile=True)
    G_PI_avg, hdr_G_PI_avg = make_PI_file(directory,G_Q_list,G_U_list,hdr_G_Q_list,
                                           avg_order='PI_of_mean', keep_PI=False, savefile=True)
    
    C_avg_PI, hdr_C_avg_PI = make_PI_file(directory,C_Q_list,C_U_list,hdr_C_Q_list,
                                       avg_order='mean_of_PI', keep_PI=False, savefile=True)
    C_PI_avg, hdr_C_PI_avg = make_PI_file(directory,C_Q_list,C_U_list,hdr_C_Q_list,
                                       avg_order='PI_of_mean', keep_PI=False, savefile=True)
    
    CG_avg_PI, hdr_CG_avg_PI = make_PI_file(directory,CG_Q_list,CG_U_list,hdr_CG_Q_list,
                                   avg_order='mean_of_PI', keep_PI=False, savefile=True)
    CG_PI_avg, hdr_CG_PI_avg = make_PI_file(directory,CG_Q_list,CG_U_list,hdr_CG_Q_list,
                                   avg_order='PI_of_mean', keep_PI=False, savefile=True)
    

## Get noise estimates:

In [ ]:
dir_in = '/srv/data/cgps-gmims/conv_regrid/'
idxs = [1,2,3,4]

fig,axs = plt.subplots(3,1,figsize=(12,8))

dQ_list, dU_list = calc_errors(dir_in, [88, 57, 173], [-2, -4.7, 0], 'CG_conv4_regrd',boxsize = 0.5)
for i in range(0,3):
    axs[0].scatter(idxs,dQ_list[i],color='C0',label='Stokes Q CGPS+GMIMS')
    axs[0].scatter(idxs,dU_list[i],color='C1',label='Stokes U CGPS+GMIMS')
    if i == 0:
        axs[0].legend()
        axs[0].grid()

dQ_list, dU_list = calc_errors(dir_in, [88, 57, 173], [-2, -4.7, 0], 'C_conv4_regrd',boxsize = 0.5)
for i in range(0,3):
    axs[1].scatter(idxs,dQ_list[i],color='C0',label='Stokes Q CGPS')
    axs[1].scatter(idxs,dU_list[i],color='C1',label='Stokes U CGPS')
    if i == 0:
        axs[1].legend()
        axs[1].grid()

dQ_list, dU_list = calc_errors(dir_in, [88, 57, 173], [-2, -4.7, 0], 'G_regrd',boxsize = 0.5)
for i in range(0,3):
    axs[2].scatter(idxs,dQ_list[i],color='C0',label='Stokes Q GMIMS')
    axs[2].scatter(idxs,dU_list[i],color='C1',label='Stokes U GMIMS')
    if i == 0:
        axs[2].legend()
        axs[2].grid()

for j in range(0,3):
    axs[j].set_ylim(0,0.1)

## Calculate polarization angles and errors

In [ ]:
%%time
go = True

dir_in  = '/srv/data/cgps-gmims/conv_regrid/'

if go:

    G_Q_list, hdr_G_Q_list = read_files_4channels(dir_in,'Q','G_regrd')
    G_U_list, hdr_G_U_list = read_files_4channels(dir_in,'U','G_regrd')
    
    C_Q_list, hdr_C_Q_list = read_files_4channels(dir_in,'Q','C_conv4_regrd')
    C_U_list, hdr_C_U_list = read_files_4channels(dir_in,'U','C_conv4_regrd')
    
    CG_Q_list, hdr_CG_Q_list = read_files_4channels(dir_in,'Q','CG_conv4_regrd')
    CG_U_list, hdr_CG_U_list = read_files_4channels(dir_in,'U','CG_conv4_regrd')
    
    PA_list_G, hdrPA_list_G = calc_pol_angles(G_Q_list,G_U_list,
                                              hdr_G_Q_list,hdr_G_U_list,0.04,0.04,
                                              keep_PA=False,savefile=True)

    PA_list_C, hdrPA_list_C = calc_pol_angles(C_Q_list,C_U_list,
                                              hdr_C_Q_list,hdr_C_U_list,0.04,0.04,
                                              keep_PA=False,savefile=True)

    PA_list_CG, hdrPA_list_CG = calc_pol_angles(CG_Q_list,CG_U_list,
                                              hdr_CG_Q_list,hdr_CG_U_list,0.04,0.04,
                                              keep_PA=False,savefile=True)

## Calculate RMs and errors

In [ ]:
%%time
go = True

dir_in  = '/srv/data/cgps-gmims/conv_regrid/'

if go:

    PA_CG_list = []
    PA_G_list = []
    PA_C_list = []
    
    band = ['A','B','C','D']
    bandlc = ['a','b','c','d']

    PA_C_err_list = []
    PA_G_err_list = []
    PA_CG_err_list = []
    
    for i in range(0,4):
    
        print('band '+band[i])
        
        # CGPS + GMIMS (CG)
        directory = '/srv/data/cgps-gmims/conv_regrid/'
        hdu_CG = fits.open(directory+'PA_'+band[i]+'_CG_conv4_regrd.fits')
        PA_CG_list.append(hdu_CG[0].data)
        PA_CG_err_list.append(hdu_CG[1].data)
        hdr_CG = hdu_CG[0].header
        
        # CGPS only (C)
        directory = '/srv/data/cgps-gmims/conv_regrid/'
        hdu_C = fits.open(directory+'PA_'+band[i]+'_C_conv4_regrd.fits')
        PA_C_list.append(hdu_C[0].data)
        PA_C_err_list.append(hdu_C[1].data)
        hdr_C = hdu_C[0].header
        
        # GMIMS only (G)
        directory = '/srv/data/cgps-gmims/conv_regrid/'
        hdu_G = fits.open(directory+'PA_'+band[i]+'_G_regrd.fits')
        PA_G_list.append(hdu_G[0].data)
        PA_G_err_list.append(hdu_G[1].data)
        hdr_G = hdu_G[0].header

    RM_CG,hdu_list_CG = calc_RM(PA_CG_list, PA_CG_err_list, hdr_CG, keep_RM=False,savefile=True)
    RM_C,hdu_list_C = calc_RM(PA_C_list, PA_C_err_list, hdr_C, keep_RM=False,savefile=True)
    RM_G,hdu_list_G = calc_RM(PA_G_list, PA_G_err_list, hdr_G, keep_RM=False,savefile=True)


## Calculate noise maps

### CGPS+GMIMS

In [ ]:
map_list = []

dir_in = '/srv/data/cgps-gmims/conv_regrid/'

files = [dir_in+'QA_CG_conv4_regrd.fits',
         dir_in+'QB_CG_conv4_regrd.fits',
         dir_in+'QC_CG_conv4_regrd.fits',
         dir_in+'QD_CG_conv4_regrd.fits',
         dir_in+'UA_CG_conv4_regrd.fits',
         dir_in+'UB_CG_conv4_regrd.fits',
         dir_in+'UC_CG_conv4_regrd.fits',
         dir_in+'UD_CG_conv4_regrd.fits']

print(files)

for file in files:

    map_list.append(fits.open(file)[0].data)


hdr = fits.open(file)[0].header

print(repr(hdr))
print('')

dpix = int(5*(3/60)/hdr['CDELT2']) # beams*(deg/beam)/(deg/pixel)
print(hdr['CDELT2'],' deg per pixel')
print(dpix,' pixels per std calc')

noise_maps = noise_map(map_list,hdr,dpix)

In [ ]:
dir_in = '/srv/data/cgps-gmims/conv_regrid/'

files = [dir_in+'QA_CG_conv4_regrd_noise.fits',
         dir_in+'QB_CG_conv4_regrd_noise.fits',
         dir_in+'QC_CG_conv4_regrd_noise.fits',
         dir_in+'QD_CG_conv4_regrd_noise.fits',
         dir_in+'UA_CG_conv4_regrd_noise.fits',
         dir_in+'UB_CG_conv4_regrd_noise.fits',
         dir_in+'UC_CG_conv4_regrd_noise.fits',
         dir_in+'UD_CG_conv4_regrd_noise.fits']

mapsout = ['qa','qb','qc','qd','ua','ub','uc','ud']

for i in range(0,8):

    print(files[i])
    print(mapsout[i])
    print('')

    fits.writeto(files[i],noise_maps[mapsout[i]],header=hdr,overwrite=True)

### CGPS only

In [ ]:
map_list = []

dir_in = '/srv/data/cgps-gmims/conv_regrid/'

files = [dir_in+'QA_C_conv4_regrd.fits',
         dir_in+'QB_C_conv4_regrd.fits',
         dir_in+'QC_C_conv4_regrd.fits',
         dir_in+'QD_C_conv4_regrd.fits',
         dir_in+'UA_C_conv4_regrd.fits',
         dir_in+'UB_C_conv4_regrd.fits',
         dir_in+'UC_C_conv4_regrd.fits',
         dir_in+'UD_C_conv4_regrd.fits']

print(files)

for file in files:

    map_list.append(fits.open(file)[0].data)


hdr = fits.open(file)[0].header

print(repr(hdr))
print('')

dpix = int(5*(3/60)/hdr['CDELT2']) # beams*(deg/beam)/(deg/pixel)
print(hdr['CDELT2'],' deg per pixel')
print(dpix,' pixels per std calc')

noise_maps = noise_map(map_list,hdr,dpix)

In [ ]:
dir_in = '/srv/data/cgps-gmims/conv_regrid/'

files = [dir_in+'QA_C_conv4_regrd_noise.fits',
         dir_in+'QB_C_conv4_regrd_noise.fits',
         dir_in+'QC_C_conv4_regrd_noise.fits',
         dir_in+'QD_C_conv4_regrd_noise.fits',
         dir_in+'UA_C_conv4_regrd_noise.fits',
         dir_in+'UB_C_conv4_regrd_noise.fits',
         dir_in+'UC_C_conv4_regrd_noise.fits',
         dir_in+'UD_C_conv4_regrd_noise.fits']

mapsout = ['qa','qb','qc','qd','ua','ub','uc','ud']

for i in range(0,8):

    print(files[i])
    print(mapsout[i])
    print('')

    fits.writeto(files[i],noise_maps[mapsout[i]],header=hdr,overwrite=True)